# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Houssem-Bjaoui/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Plain-language data contract (Lane 2: Refresh / Content Opportunity Scoring)**

- **One row means:** one `client_hash_id` × `content_hash_id` page snapshot on one `report_date` from the daily fact table.
- **Table(s) used:** `fact_content_daily_performance` (primary signals + proxy trend fields), joined with `dim_content` (content metadata like `word_count`).
- **Time window used:** development month **March 2026** (`2026-03-01` to `2026-03-31`), with a decision snapshot on `2026-03-31` for quick scoring.
- **What is predicted/ranked:** a decision-support ranking of pages likely to need refresh review, using proxy label `trend_direction = 'down'` at the March snapshot.
- **One deliberate exclusion:** June 2026 (`fact_content_daily_performance_sample`) is excluded from label logic because it is the sealed final month; using it in development would leak evaluation context.


In [27]:
import os
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    print(
        "HF_TOKEN not found. Add it as a Colab Secret, "
        "then Run All again."
    )
    HAS_WAREHOUSE = False
else:
    HAS_WAREHOUSE = True

if HAS_WAREHOUSE:
    con = duckdb.connect()

    con.execute(
        f"CREATE OR REPLACE SECRET hf "
        f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
    )

    REL = "hf://datasets/FlyRank/internship-warehouse"

    TABLES = {
        "dim_content": (
            f"read_parquet('{REL}/dim_content.parquet')"
        ),
        "fact_daily": (
            f"read_parquet("
            f"'{REL}/fact_content_daily_performance/**/*.parquet'"
            f")"
        ),
    }

    DEV_MONTH = "2026-03"

    print("Connected.")
    print("Development month:", DEV_MONTH)

Connected.
Development month: 2026-03


In [28]:
if HAS_WAREHOUSE:
    print("=== fact_daily ===")
    print(
        con.sql(f"""
            DESCRIBE SELECT *
            FROM {TABLES['fact_daily']}
            LIMIT 1
        """).df()[["column_name", "column_type"]].to_string(index=False)
    )

    print("\n=== dim_content ===")
    print(
        con.sql(f"""
            DESCRIBE SELECT *
            FROM {TABLES['dim_content']}
            LIMIT 1
        """).df()[["column_name", "column_type"]].to_string(index=False)
    )

=== fact_daily ===
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_co

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature fields (max 5, honest)
1. `impressions_prev_30d` (feature: demand baseline)  
   **Knowable at the decision moment because** it is measured in the prior 30-day window already completed by `report_date`.
2. `clicks_prev_30d` (feature: click demand baseline)  
   **Knowable at the decision moment because** it is observed before or at the same snapshot date.
3. `gsc_avg_position` (feature: search visibility context)  
   **Knowable at the decision moment because** it is an observed ranking signal already logged in the warehouse by the snapshot date.
4. `days_since_last_update` (feature: freshness recency)  
   **Knowable at the decision moment because** content update recency is known immediately on that day.
5. `word_count` (feature: content depth proxy)  
   **Knowable at the decision moment because** page length metadata exists at decision time and does not require future outcomes.

### Label / proxy field
- `is_down_proxy = 1(trend_direction = 'down')` at `report_date='2026-03-31'`.

### Context fields (not model inputs)
- `client_hash_id`, `content_hash_id`, `report_date`.

### Excluded fields
- `trend_direction` and `trend_pct` are excluded as honest features because the label is derived from the same trend logic.
- `fact_content_daily_performance_sample` (June 2026) is excluded from development label logic because it is the sealed final month.


In [29]:
if HAS_WAREHOUSE:

    snapshot = con.sql(f"""
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.month,

            f.gsc_impressions,
            f.gsc_clicks,
            f.gsc_avg_position,

            f.gsc_data_available,
            f.ga4_data_available,

            d.word_count,
            d.content_type,
            d.content_created_date,
            d.content_updated_date,
            d.is_published,
            d.is_deleted

        FROM {TABLES['fact_daily']} f

        LEFT JOIN {TABLES['dim_content']} d
            ON f.content_hash_id = d.content_hash_id
            AND f.client_hash_id = d.client_hash_id

        WHERE f.month = '{DEV_MONTH}'
    """).df()

    print("Rows in March 2026:", len(snapshot))

    display(snapshot.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in March 2026: 9841378


,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,ga4_data_available,word_count,content_type,content_created_date,content_updated_date,is_published,is_deleted
0,client_2094c6eb080311d5,content_149355c8dfc3f8e1,2026-03,0,0,NaN,False,False,4038,keyword article,2026-01-22,2026-05-12,True,False
1,client_2094c6eb080311d5,content_14a3d47ccd0d15dc,2026-03,0,0,NaN,False,False,3864,keyword article,2025-12-09,2026-05-12,True,False
2,client_2094c6eb080311d5,content_14a6f92117604fef,2026-03,2,0,4.5,True,False,2867,keyword article,2025-12-11,2026-05-20,True,False
3,client_2094c6eb080311d5,content_14a86c63a214f648,2026-03,2,0,75.0,True,False,2949,keyword article,2025-12-09,2026-05-12,True,False
4,client_2094c6eb080311d5,content_14b1a02c1b8557fb,2026-03,0,0,NaN,False,False,4279,keyword article,2025-12-11,2026-05-12,True,False


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Exactly three verification queries are used below:
1. Grain check
2. Row count + date span check
3. Availability check using `IS TRUE`


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if HAS_WAREHOUSE:
    q1 = f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS dup_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{DEV_START}' AND DATE '{DEV_END}'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
    """

    q2 = f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{DEV_START}' AND DATE '{DEV_END}'
    """

    q3 = f"""
    SELECT
        COUNT(*) AS rows_total,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / NULLIF(COUNT(*), 0),
            2
        ) AS pct_ga4_available
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{DEV_START}' AND DATE '{DEV_END}'
    """

    print('Query 1 — verify grain (expect empty result):')
    display(con.sql(q1).df())

    print('Query 2 — verify row count and date span:')
    display(con.sql(q2).df())

    print('Query 3 — verify availability using IS TRUE:')
    display(con.sql(q3).df())


Query 1 — verify grain (expect empty result):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,dup_rows


Query 2 — verify row count and date span:


,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


Query 3 — verify availability using IS TRUE:


,rows_total,rows_ga4_available,pct_ga4_available
0,9841378,413966,4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One important limitation of this selected slice: **March 2026 is one development month from an unbalanced panel**, so behavior can differ by client history depth and measurement coverage. This means scores are useful for decision-support ranking, but they are not causal proof that refreshing content will cause recovery.


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
if HAS_WAREHOUSE:
    limit_check = con.sql(f"""
        SELECT
            COUNT(*) AS clients_total,
            MIN(gsc_data_start) AS min_gsc_start,
            MAX(gsc_data_start) AS max_gsc_start,
            MIN(ga4_data_start) AS min_ga4_start,
            MAX(ga4_data_start) AS max_ga4_start
        FROM read_parquet('{REL}/dim_clients.parquet')
    """).df()
    display(limit_check)


,clients_total,min_gsc_start,max_gsc_start,min_ga4_start,max_ga4_start
0,104,2025-01-27,2026-06-02,2025-10-29,2026-06-01


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
